# 🇮🇱 Israeli Data RAG System
### LangChain + Pinecone + data.gov.il

This notebook builds a RAG (Retrieval-Augmented Generation) system that:
1. Fetches Israeli localities/cities data from **data.gov.il**
2. Chunks & embeds the data into **Pinecone** vector store
3. Uses **LangChain + OpenAI** to answer natural language questions about the data

Example questions you can ask:
- *"What are the largest cities in Israel by population?"*
- *"Which cities are in the Northern district?"*
- *"Tell me about Tel Aviv"*

## Step 1: Install Dependencies

In [1]:
! pip install -q langchain langchain-openai langchain-pinecone langchain-community pinecone openai python-dotenv requests pandas tiktoken requests

## Step 2: Imports & Configuration

In [2]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_pinecone import PineconeVectorStore
from langchain_core.documents import Document
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate
from pinecone import Pinecone, ServerlessSpec

load_dotenv(".env")

# Clear Intel proxy env vars — they break requests when not on VPN
for _var in ("HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy",
             "NO_PROXY", "no_proxy"):
    os.environ.pop(_var, None)

OPENAI_API_KEY   = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
INDEX_NAME       = "israel-gov-data"

print("✅ Keys loaded")
print(f"   OpenAI:   {'✅ set' if OPENAI_API_KEY else '❌ missing — set OPENAI_API_KEY in .env'}")
print(f"   Pinecone: {'✅ set' if PINECONE_API_KEY else '❌ missing — set PINECONE_API_KEY in .env'}")

c:\Users\hmoshayo\PycharmProjects\AIExperts\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Keys loaded
   OpenAI:   ✅ set
   Pinecone: ✅ set


## Step 3: Fetch Wikipedia Articles on Israeli Cities
We fetch the **Wikipedia article extract** (full introductory section) for a curated list of Israeli cities and towns.  
Each article gives rich, descriptive text about history, geography, demographics, and culture — ideal for a RAG system.

In [3]:
ISRAELI_CITIES = [
    "Tel Aviv", "Jerusalem", "Haifa", "Rishon LeZion", "Petah Tikva",
    "Ashdod", "Netanya", "Beer Sheva", "Bnei Brak", "Holon",
    "Bat Yam", "Rehovot", "Ashkelon", "Beit Shemesh", "Kfar Saba",
    "Herzliya", "Ra'anana", "Nazareth", "Modi'in-Maccabim-Re'ut", "Acre",
    "Eilat", "Tiberias", "Safed", "Nahariya", "Lod", "Ramla",
    "Givatayim", "Ramat Gan", "Or Yehuda", "Ness Ziona",
    "Kiryat Ata", "Kiryat Gat", "Dimona", "Arad", "Mitzpe Ramon",
    "Kiryat Shmona", "Afula", "Hadera", "Netivot", "Sderot",
]

WIKI_API = "https://en.wikipedia.org/w/api.php"
HEADERS = {"User-Agent": "IsraeliCitiesRAG/1.0 (educational project; contact@example.com)"}

def fetch_wikipedia_article(title: str) -> dict | None:
    """Fetch the full introductory extract of a Wikipedia article."""
    params = {
        "action": "query",
        "prop": "extracts",
        "exintro": True,
        "explaintext": True,
        "redirects": True,
        "titles": title,
        "format": "json",
    }
    try:
        resp = requests.get(WIKI_API, params=params, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        pages = resp.json()["query"]["pages"]
        page = next(iter(pages.values()))
        if "missing" in page or not page.get("extract", "").strip():
            return None
        return {"title": page["title"], "text": page["extract"].strip()}
    except Exception as e:
        print(f"  ⚠️  Could not fetch '{title}': {e}")
        return None

articles = []
print(f"Fetching {len(ISRAELI_CITIES)} Wikipedia articles...\n")
for city in ISRAELI_CITIES:
    art = fetch_wikipedia_article(city)
    if art:
        articles.append(art)
        print(f"  ✅ {art['title']}  ({len(art['text'])} chars)")
    else:
        print(f"  ⚠️  Skipped: {city}")

print(f"\n✅ Fetched {len(articles)} articles")
print(f"Average length: {sum(len(a['text']) for a in articles) // len(articles)} chars")

Fetching 40 Wikipedia articles...

  ✅ Tel Aviv  (2895 chars)
  ✅ Jerusalem  (4401 chars)
  ✅ Haifa  (2579 chars)
  ✅ Rishon LeZion  (744 chars)
  ✅ Petah Tikva  (707 chars)
  ✅ Ashdod  (1511 chars)
  ✅ Netanya  (923 chars)
  ✅ Beersheba  (2432 chars)
  ✅ Bnei Brak  (400 chars)
  ✅ Holon  (325 chars)
  ✅ Bat Yam  (210 chars)
  ✅ Rehovot  (215 chars)
  ✅ Ashkelon  (1858 chars)
  ✅ Beit Shemesh  (474 chars)
  ✅ Kfar Saba  (316 chars)
  ✅ Herzliya  (623 chars)
  ✅ Ra'anana  (730 chars)
  ✅ Nazareth  (2113 chars)
  ✅ Modi'in-Maccabim-Re'ut  (1082 chars)
  ✅ Acre  (1235 chars)
  ✅ Eilat  (1673 chars)
  ✅ Tiberias  (2238 chars)
  ✅ Safed  (4207 chars)
  ✅ Nahariya  (188 chars)
  ✅ Lod  (2046 chars)
  ✅ Ramla  (1856 chars)
  ✅ Givatayim  (605 chars)
  ✅ Ramat Gan  (484 chars)
  ✅ Or Yehuda  (179 chars)
  ✅ Ness Ziona  (192 chars)
  ✅ Kiryat Ata  (191 chars)
  ✅ Kiryat Gat  (422 chars)
  ✅ Dimona  (424 chars)
  ✅ Arad  (26 chars)
  ✅ Mitzpe Ramon  (327 chars)
  ✅ Kiryat Shmona  (297 chars)
  ✅

## Step 4: Convert Articles to LangChain Documents
Each Wikipedia article becomes a LangChain `Document` with the article text as content and the city name as metadata.

In [4]:
raw_docs = [
    Document(
        page_content=article["text"],
        metadata={"source": "wikipedia", "city": article["title"]},
    )
    for article in articles
]

print(f"✅ Created {len(raw_docs)} documents")
print("\n--- Sample document ---")
print(f"City: {raw_docs[0].metadata['city']}")
print(raw_docs[0].page_content[:600])

✅ Created 40 documents

--- Sample document ---
City: Tel Aviv
Tel Aviv, officially Tel Aviv-Yafo, and also known as Tel Aviv-Jaffa, is the most populous city in the Gush Dan metropolitan area of Israel. Located on the Israeli Mediterranean coastline and with a population of 495,230, it is the economic and technological center of the country and a global high-tech hub. If East Jerusalem is considered part of Israel, Tel Aviv is the country's second-most-populous city, after Jerusalem; if not, Tel Aviv is the most populous city, ahead of West Jerusalem.
Tel Aviv is governed by the Tel Aviv-Yafo Municipality, headed by Mayor Ron Huldai, and is home to most 


## Step 5: Split Documents into Chunks

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " "],
)

chunks = splitter.split_documents(raw_docs)
print(f"✅ Split into {len(chunks)} chunks (from {len(raw_docs)} documents)")
print(f"\n--- Sample chunk ---")
print(chunks[0].page_content)

✅ Split into 128 chunks (from 40 documents)

--- Sample chunk ---
Tel Aviv, officially Tel Aviv-Yafo, and also known as Tel Aviv-Jaffa, is the most populous city in the Gush Dan metropolitan area of Israel. Located on the Israeli Mediterranean coastline and with a population of 495,230, it is the economic and technological center of the country and a global high-tech hub. If East Jerusalem is considered part of Israel, Tel Aviv is the country's second-most-populous city, after Jerusalem; if not, Tel Aviv is the most populous city, ahead of West Jerusalem.


## Step 6: Create Pinecone Index & Upload Embeddings
Creates the index if it doesn't exist, then upsets all document chunks.  
⚠️ **Only run this once** — re-running will re-upload duplicates. Delete the index first on [pinecone.io](https://app.pinecone.io) if you want to start fresh.

In [6]:
# Initialize Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

# Create index if it doesn't exist
existing_indexes = [idx.name for idx in pc.list_indexes()]
if INDEX_NAME not in existing_indexes:
    print(f"Creating index '{INDEX_NAME}'...")
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,          # text-embedding-3-small dimension
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print("✅ Index created")
else:
    print(f"✅ Index '{INDEX_NAME}' already exists")

# Initialize embeddings model
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=OPENAI_API_KEY,
)

# Upload all chunks to Pinecone
print(f"\nUploading {len(chunks)} chunks to Pinecone...")
vector_store = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=INDEX_NAME,
    pinecone_api_key=PINECONE_API_KEY,
)
print("✅ All chunks uploaded to Pinecone!")

✅ Index 'israel-gov-data' already exists

Uploading 128 chunks to Pinecone...
✅ All chunks uploaded to Pinecone!


## Step 7: Build the RAG Chain
Connect the Pinecone retriever to an LLM to create the question-answering pipeline.

In [18]:
# Connect to the existing index (no re-upload needed after first run)
vector_store = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embeddings,
    pinecone_api_key=PINECONE_API_KEY,
)

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

# LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=OPENAI_API_KEY,
)

# Custom prompt — instructs the LLM to use only retrieved context
prompt_template = """You are an expert on Israeli cities, towns, and localities.
Use ONLY the context below (from Wikipedia) to answer the question.
If the answer is not in the context, say "I don't have that information in the dataset."

Context:
{context}

Question: {question}

Answer:"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"],
)

# Build the RAG chain
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT},
)

print("✅ RAG chain ready!")

✅ RAG chain ready!


## Step 8: Ask Questions!

In [19]:
def ask(question: str, show_sources: bool = False) -> str:
    """Ask a question to the RAG system."""
    result = rag_chain.invoke({"query": question})
    answer = result["result"]
    
    print(f"❓ {question}")
    print(f"💬 {answer}")
    
    if show_sources:
        print("\n📄 Sources:")
        for i, doc in enumerate(result["source_documents"], 1):
            print(f"  [{i}] {doc.page_content[:200]}...")
    
    print("-" * 60)
    return answer

In [20]:
# Example questions — feel free to change these!
ask("What are the largest cities in Israel by population?")
ask("Which localities are in the Northern district?")
ask("Tell me about Tel Aviv")
ask("What is the smallest town in Israel?", show_sources=True)

❓ What are the largest cities in Israel by population?
💬 The largest cities in Israel by population are:

1. Jerusalem (if East Jerusalem is considered part of Israel)
2. Tel Aviv (if East Jerusalem is not considered part of Israel)
3. Haifa
4. Petah Tikva
5. Beersheba
6. Netanya
7. Nazareth

(Note: The specific ranking of Haifa is not provided in the context, but it is generally known to be one of the largest cities in Israel.)
------------------------------------------------------------
❓ Which localities are in the Northern district?
💬 The localities in the Northern District of Israel are Nazareth and Kiryat Shmona.
------------------------------------------------------------
❓ Tell me about Tel Aviv
💬 Tel Aviv, officially known as Tel Aviv-Yafo and also referred to as Tel Aviv-Jaffa, is the most populous city in the Gush Dan metropolitan area of Israel, with a population of 495,230. It is located on the Israeli Mediterranean coastline and serves as the economic and technological ce

"I don't have that information in the dataset."

## Your Turn — Ask Anything!

In [23]:
my_question = "Where is Givatayim?"
ask(my_question, show_sources=True)

❓ Where is Givatayim?
💬 Givatayim is a city in Israel east of Tel Aviv.

📄 Sources:
  [1] Givatayim (Hebrew: גבעתיים, lit. 'Two hills') is a city in Israel east of Tel Aviv. It is part of the Gush Dan metropolitan area. Givatayim was established in 1922 by pioneers of the Second Aliyah. In...
  [2] Kiryat Gat (Hebrew: קריית גת, lit. 'City of Gat') also spelled Qiryat Gat, is a city in the Southern District of Israel. It lies 56 km (35 miles) south of Tel Aviv, 43 km (27 mi) north of Beersheba, a...
  [3] Bat Yam (Hebrew: בת ים ) is a city in the Tel Aviv District of Israel, on the Central Coastal Plain just south of Tel Aviv. It is part of the Gush Dan metropolitan area. In 2023, it had a population o...
  [4] Netanya (Hebrew: נתניה, IPA: [netanˈja]), or Natanya (IPA: [naˈtanja]), is a city in the Central District of Israel, and is the capital of the surrounding Sharon plain. It is 30 kilometres (20 mi) nor...
  [5] Netivot (Hebrew: נתיבות, lit. 'Paths') is a city located in the Southe

'Givatayim is a city in Israel east of Tel Aviv.'